In [12]:
import os
import re
import pandas as pd

from socceraction.data.statsbomb import StatsBombLoader
import socceraction.spadl.statsbomb as sb_spadl
import socceraction.spadl as spadl
import socceraction.spadl.config as spadlcfg
import socceraction.vaep.features as vaep_features
import socceraction.vaep.labels as vaep_labels

pd.set_option('display.max_rows', 10)


import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [13]:
def _slug(text: str) -> str:
    text = str(text).strip().lower()
    text = text.replace('/', '_').replace('\\', '_')
    text = re.sub(r'[^a-z0-9]+', '_', text)
    text = re.sub(r'_+', '_', text).strip('_')
    return text

def resolve_competition_season_ids(df: pd.DataFrame, competition_name: str, season_name: str) -> tuple[int, int]:
    m = df[(df['competition_name'] == competition_name) & (df['season_name'] == season_name)]
    if len(m) == 0:
        raise ValueError(f"No match for competition={competition_name!r}, season={season_name!r}")
    if len(m) > 1:
        opts = m[['competition_id', 'season_id', 'competition_name', 'season_name']].drop_duplicates().to_dict('records')
        raise ValueError(
            f"Ambiguous match for competition={competition_name!r}, season={season_name!r}. "
            f"Use selected_id_pairs instead. Options: {opts}"
        )
    row = m.iloc[0]
    return int(row.competition_id), int(row.season_id)



def _as_series(obj, name: str) -> pd.Series:
    """Compatibility shim: socceraction labels may return Series or 1-col DataFrame."""
    if isinstance(obj, pd.DataFrame):
        if name in obj.columns and obj.shape[1] == 1:
            s = obj[name]
        elif obj.shape[1] == 1:
            s = obj.iloc[:, 0]
        else:
            s = obj.iloc[:, 0]
    else:
        s = obj
    return s.rename(name)

def build_and_save_vaep_for_comp_season(competition_id: int, season_id: int) -> tuple[str, str]:
    # Resolve names (for file naming)
    row = df_competitions[(df_competitions.competition_id == competition_id) & (df_competitions.season_id == season_id)]
    if len(row) != 1:
        raise ValueError(f'Cannot uniquely resolve competition_id={competition_id}, season_id={season_id}')
    comp_name = str(row.iloc[0].competition_name)
    season_name = str(row.iloc[0].season_name)

    league_slug = _slug(comp_name)
    season_slug = _slug(season_name)

    features_path = os.path.join(output_dir, f'features_{league_slug}_{season_slug}.h5')
    labels_path = os.path.join(output_dir, f'labels_{league_slug}_{season_slug}.h5')

    print(f'\n=== {comp_name} | {season_name} (cid={competition_id}, sid={season_id}) ===')
    print('->', features_path)
    print('->', labels_path)

    games = SBL.games(competition_id=competition_id, season_id=season_id).copy()
    if games.empty:
        raise ValueError('No games found for this competition/season')

    # Convert events -> SPADL actions for each game
    actions_parts = []
    failed = []
    for i, game in games.reset_index(drop=True).iterrows():
        game_id = int(game.game_id)
        home_team_id = int(game.home_team_id)
        try:
            events = SBL.events(game_id)
            actions = sb_spadl.convert_to_actions(
                events,
                home_team_id=home_team_id,
                xy_fidelity_version=1,
                shot_fidelity_version=1,
            )
            actions_parts.append(actions)
        except Exception as e:
            failed.append((game_id, str(e)))

        if (i + 1) % 50 == 0:
            print(f'converted {i+1}/{len(games)} games')

    if failed:
        print('Failed games:', len(failed))
        display(pd.DataFrame(failed, columns=['game_id','error']).head(10))

    if not actions_parts:
        raise ValueError('No actions were generated (all games failed?)')

    actions = pd.concat(actions_parts, ignore_index=True)

    # Ensure action_id exists per game
    if 'action_id' not in actions.columns:
        actions = actions.sort_values(['game_id', 'period_id', 'time_seconds']).copy()
        actions['action_id'] = actions.groupby('game_id').cumcount().astype(int)

    # Some versions of socceraction.vaep.features require type_name
    if 'type_name' not in actions.columns and 'type_id' in actions.columns:
        _actiontype_map = dict(enumerate(spadlcfg.actiontypes))
        actions['type_name'] = actions['type_id'].map(_actiontype_map).astype(str)

    # Normalize direction (play left-to-right for home team)
    actions = actions.merge(games[['game_id', 'home_team_id']], on='game_id', how='left')
    actions = actions.sort_values(['game_id', 'period_id', 'time_seconds', 'action_id']).reset_index(drop=True)
    actions = actions.groupby('game_id', group_keys=False).apply(
        lambda x: spadl.play_left_to_right(
            x.drop(columns=['home_team_id']),
            home_team_id=int(x.home_team_id.iloc[0]),
        )
    ).reset_index(drop=True)

    nb_prev_actions = 3
    xfns = [
        vaep_features.result_onehot,
        vaep_features.startlocation,
        vaep_features.movement,
        vaep_features.goalscore,
    ]

    features_parts = []
    labels_parts = []

    for game_id in games.game_id.astype(int).tolist():
        game_actions = actions[actions.game_id == game_id].reset_index(drop=True)
        if len(game_actions) == 0:
            continue

        gs = vaep_features.gamestates(game_actions, nb_prev_actions)
        X_game = pd.concat([fn(gs) for fn in xfns], axis=1)
        X_game.insert(0, 'game_id', game_id)
        X_game.insert(1, 'action_id', game_actions.action_id.astype(int).values)
        features_parts.append(X_game)

        y_scores = _as_series(vaep_labels.scores(game_actions), 'scores')
        y_concedes = _as_series(vaep_labels.concedes(game_actions), 'concedes')
        Y_game = pd.concat([y_scores, y_concedes], axis=1)
        Y_game.insert(0, 'game_id', game_id)
        Y_game.insert(1, 'action_id', game_actions.action_id.astype(int).values)
        labels_parts.append(Y_game)

    df_features = pd.concat(features_parts, ignore_index=True) if features_parts else pd.DataFrame()
    df_labels = pd.concat(labels_parts, ignore_index=True) if labels_parts else pd.DataFrame()

    # Overwrite outputs
    for f in [features_path, labels_path]:
        if os.path.exists(f):
            os.remove(f)

    with pd.HDFStore(features_path, mode='w') as store:
        store.put('features', df_features, format='table')

    with pd.HDFStore(labels_path, mode='w') as store:
        store.put('labels', df_labels, format='table')

    print('Saved features:', df_features.shape)
    print('Saved labels:  ', df_labels.shape)

    return features_path, labels_path

In [14]:
# Paths
data_root = '../data'  # StatsBomb open-data JSON folder
output_dir = os.path.join('..', 'data', 'spadl_data')
os.makedirs(output_dir, exist_ok=True)

# Loader
SBL = StatsBombLoader(getter='local', root=data_root)

print('output_dir:', output_dir)

output_dir: data\spadl_data


In [15]:
# List available competitions/seasons
df_competitions = SBL.competitions().copy()
cols = ['competition_id', 'season_id', 'competition_name', 'season_name']
display(df_competitions[cols].sort_values(['competition_name', 'season_name']).reset_index(drop=True))

,competition_id,season_id,competition_name,season_name
0,9,27,1. Bundesliga,2015/2016
1,9,281,1. Bundesliga,2023/2024
2,1267,107,African Cup of Nations,2023
3,16,276,Champions League,1970/1971
4,16,71,Champions League,1971/1972
...,...,...,...,...
70,35,75,UEFA Europa League,1988/1989
71,53,106,UEFA Women's Euro,2022
72,53,315,UEFA Women's Euro,2025
73,72,30,Women's World Cup,2019


## Select league/season

Edit `selected_name_pairs` to include the league(s) and season(s) you want.

- The `competition_name` and `season_name` must match exactly the values shown in the table above.
- If there are duplicates, use `selected_id_pairs` instead (competition_id, season_id).

In [16]:
# Select league/season pairs to process
SAVE_ALL_AVAILABLE = True  # True => process every available competition+season in df_competitions

# OPTION A: select by names (used only when SAVE_ALL_AVAILABLE = False)
selected_name_pairs = [
    # ('Serie A', '2015/2016'),
]

# OPTION B: select by ids (used only when SAVE_ALL_AVAILABLE = False)
selected_id_pairs = [
    # (11, 27),
]

if SAVE_ALL_AVAILABLE:
    selected = [
        (int(row.competition_id), int(row.season_id))
        for row in df_competitions[['competition_id', 'season_id']].drop_duplicates().itertuples(index=False)
    ]
elif selected_name_pairs and selected_id_pairs:
    raise ValueError('Use either selected_name_pairs OR selected_id_pairs (not both).')
elif selected_name_pairs:
    selected = [resolve_competition_season_ids(df_competitions, c, s) for (c, s) in selected_name_pairs]
elif selected_id_pairs:
    selected = [(int(c), int(s)) for (c, s) in selected_id_pairs]
else:
    selected = []

print('Total selected (competition_id, season_id):', len(selected))
print('Preview:', selected[:10])

Total selected (competition_id, season_id): 75
Preview: [(9, 281), (9, 27), (1267, 107), (16, 4), (16, 1), (16, 2), (16, 27), (16, 26), (16, 25), (16, 24)]


In [17]:
# Build and save one dataset per selected competition/season
if not selected:
    raise ValueError('No league/season selected. Configure SAVE_ALL_AVAILABLE or selection lists above.')

outputs = []
failed = []

for competition_id, season_id in selected:
    try:
        outputs.append(build_and_save_vaep_for_comp_season(competition_id, season_id))
    except Exception as e:
        failed.append((competition_id, season_id, str(e)))
        print(f'Skipped (cid={competition_id}, sid={season_id}): {e}')

print('\nDone.')
print('Saved datasets:', len(outputs))
print('Failed datasets:', len(failed))

if outputs:
    print('\nSaved files:')
    for fpath, lpath in outputs:
        print('-', fpath)
        print('-', lpath)

if failed:
    print('\nFirst failed items:')
    display(pd.DataFrame(failed, columns=['competition_id', 'season_id', 'error']).head(10))


=== 1. Bundesliga | 2023/2024 (cid=9, sid=281) ===
-> data\spadl_data\features_1_bundesliga_2023_2024.h5
-> data\spadl_data\labels_1_bundesliga_2023_2024.h5
Saved features: (80212, 38)
Saved labels:   (80212, 4)

=== 1. Bundesliga | 2015/2016 (cid=9, sid=27) ===
-> data\spadl_data\features_1_bundesliga_2015_2016.h5
-> data\spadl_data\labels_1_bundesliga_2015_2016.h5
converted 50/306 games
converted 100/306 games
converted 150/306 games
converted 200/306 games
converted 250/306 games
converted 300/306 games
Saved features: (596336, 38)
Saved labels:   (596336, 4)

=== African Cup of Nations | 2023 (cid=1267, sid=107) ===
-> data\spadl_data\features_african_cup_of_nations_2023.h5
-> data\spadl_data\labels_african_cup_of_nations_2023.h5
converted 50/52 games
Saved features: (92441, 38)
Saved labels:   (92441, 4)

=== Champions League | 2018/2019 (cid=16, sid=4) ===
-> data\spadl_data\features_champions_league_2018_2019.h5
-> data\spadl_data\labels_champions_league_2018_2019.h5
Saved feat

In [18]:
# Train a RandomForest on ALL selected data (no split)
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Collect (features, labels) from saved files for each selected comp/season
df_all = []
for competition_id, season_id in selected:
    row = df_competitions[(df_competitions.competition_id == competition_id) & (df_competitions.season_id == season_id)]
    if len(row) != 1:
        raise ValueError(f'Cannot uniquely resolve competition_id={competition_id}, season_id={season_id}')
    comp_name = str(row.iloc[0].competition_name)
    season_name = str(row.iloc[0].season_name)
    league_slug = _slug(comp_name)
    season_slug = _slug(season_name)
    features_path = os.path.join(output_dir, f'features_{league_slug}_{season_slug}.h5')
    labels_path = os.path.join(output_dir, f'labels_{league_slug}_{season_slug}.h5')
    if not (os.path.exists(features_path) and os.path.exists(labels_path)):
        raise FileNotFoundError(
            f'Missing files for {comp_name} {season_name}. Expected:\n- {features_path}\n- {labels_path}\nRun the dataset-generation cell first.'
        )

    with pd.HDFStore(features_path, mode='r') as store:
        df_features = store['features'].copy()
    with pd.HDFStore(labels_path, mode='r') as store:
        df_labels = store['labels'].copy()

    df = df_features.merge(df_labels, on=['game_id', 'action_id'], how='inner')
    df['competition_id'] = competition_id
    df['season_id'] = season_id
    df_all.append(df)
    print(f'Loaded + joined {comp_name} {season_name}:', df.shape)

df_all = pd.concat(df_all, ignore_index=True)
print('\nTotal joined dataset:', df_all.shape)

# Build X/y (no split)
drop_cols = {'game_id', 'action_id', 'scores', 'concedes', 'competition_id', 'season_id'}
feature_cols = [c for c in df_all.columns if c not in drop_cols]
X = df_all[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y_scores = df_all['scores'].astype(int)
y_concedes = df_all['concedes'].astype(int)

print('X:', X.shape, 'y_scores:', y_scores.shape, 'y_concedes:', y_concedes.shape)

# RandomForest (baseline)
rf_scores = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced_subsample',
)
rf_concedes = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced_subsample',
)

rf_scores.fit(X, y_scores)
rf_concedes.fit(X, y_concedes)

print('Trained RandomForest models on full dataset (no split).')

Loaded + joined 1. Bundesliga 2023/2024: (80212, 42)
Loaded + joined 1. Bundesliga 2015/2016: (596336, 42)
Loaded + joined African Cup of Nations 2023: (92441, 42)
Loaded + joined Champions League 2018/2019: (1793, 42)
Loaded + joined Champions League 2017/2018: (2052, 42)
Loaded + joined Champions League 2016/2017: (2062, 42)
Loaded + joined Champions League 2015/2016: (2853, 42)
Loaded + joined Champions League 2014/2015: (2082, 42)
Loaded + joined Champions League 2013/2014: (2575, 42)
Loaded + joined Champions League 2012/2013: (1957, 42)
Loaded + joined Champions League 2011/2012: (2770, 42)
Loaded + joined Champions League 2010/2011: (2559, 42)
Loaded + joined Champions League 2009/2010: (1968, 42)
Loaded + joined Champions League 2008/2009: (2045, 42)
Loaded + joined Champions League 2006/2007: (1763, 42)
Loaded + joined Champions League 2004/2005: (2699, 42)
Loaded + joined Champions League 2003/2004: (1855, 42)
Loaded + joined Champions League 1999/2000: (2006, 42)
Loaded + jo

In [19]:
# Test set

# OPTION A: select by names (recommended UX)
# Provide a list of (competition_name, season_name) tuples.
selected_name_pairs = [
    # ('Serie A', '2015/2016'),
    ('Serie A', '1986/1987'),
 ]

# OPTION B: select by ids (uncomment to use). Provide a list of (competition_id, season_id) tuples.
selected_id_pairs = [
    # (11, 27),
 ]


if selected_name_pairs and selected_id_pairs:
    raise ValueError('Use either selected_name_pairs OR selected_id_pairs (not both).')

if selected_name_pairs:
    selected = [resolve_competition_season_ids(df_competitions, c, s) for (c, s) in selected_name_pairs]
elif selected_id_pairs:
    selected = [(int(c), int(s)) for (c, s) in selected_id_pairs]
else:
    selected = []

print('Selected (competition_id, season_id):', selected)


# Run for all selections
if not selected:
    raise ValueError('No league/season selected. Edit selected_name_pairs or selected_id_pairs above.')

outputs = []
for competition_id, season_id in selected:
    outputs.append(build_and_save_vaep_for_comp_season(competition_id, season_id))

print('\nDone. Outputs:')
for fpath, lpath in outputs:
    print('-', fpath)
    print('-', lpath)

Selected (competition_id, season_id): [(12, 86)]

=== Serie A | 1986/1987 (cid=12, sid=86) ===
-> data\spadl_data\features_serie_a_1986_1987.h5
-> data\spadl_data\labels_serie_a_1986_1987.h5
Saved features: (1711, 38)
Saved labels:   (1711, 4)

Done. Outputs:
- data\spadl_data\features_serie_a_1986_1987.h5
- data\spadl_data\labels_serie_a_1986_1987.h5


In [20]:
# Collect (features, labels) from saved files for each selected comp/season
df_all = []
for competition_id, season_id in selected:
    row = df_competitions[(df_competitions.competition_id == competition_id) & (df_competitions.season_id == season_id)]
    if len(row) != 1:
        raise ValueError(f'Cannot uniquely resolve competition_id={competition_id}, season_id={season_id}')
    comp_name = str(row.iloc[0].competition_name)
    season_name = str(row.iloc[0].season_name)
    league_slug = _slug(comp_name)
    season_slug = _slug(season_name)
    features_path = os.path.join(output_dir, f'features_{league_slug}_{season_slug}.h5')
    labels_path = os.path.join(output_dir, f'labels_{league_slug}_{season_slug}.h5')
    if not (os.path.exists(features_path) and os.path.exists(labels_path)):
        raise FileNotFoundError(
            f'Missing files for {comp_name} {season_name}. Expected:\n- {features_path}\n- {labels_path}\nRun the dataset-generation cell first.'
        )

    with pd.HDFStore(features_path, mode='r') as store:
        df_features = store['features'].copy()
    with pd.HDFStore(labels_path, mode='r') as store:
        df_labels = store['labels'].copy()

    df = df_features.merge(df_labels, on=['game_id', 'action_id'], how='inner')
    df['competition_id'] = competition_id
    df['season_id'] = season_id
    df_all.append(df)
    print(f'Loaded + joined {comp_name} {season_name}:', df.shape)

df_test = pd.concat(df_all, ignore_index=True)
print('\nTotal joined dataset:', df_test.shape)

# Build X/y (no split)
drop_cols = {'game_id', 'action_id', 'scores', 'concedes', 'competition_id', 'season_id'}
feature_cols = [c for c in df_test.columns if c not in drop_cols]
X = df_test[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y_scores = df_test['scores'].astype(int)
y_concedes = df_test['concedes'].astype(int)

print('X:', X.shape, 'y_scores:', y_scores.shape, 'y_concedes:', y_concedes.shape)


Loaded + joined Serie A 1986/1987: (1711, 42)

Total joined dataset: (1711, 42)
X: (1711, 36) y_scores: (1711,) y_concedes: (1711,)


In [21]:
# Test the trained RandomForest on this test dataset (the X/y from the previous cell)
from sklearn.metrics import (
    roc_auc_score,
    log_loss,
    brier_score_loss,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
 )

def _pos_class_proba(model, X_df):
    """Return P(class==1) for a fitted sklearn classifier."""
    proba = model.predict_proba(X_df)
    classes = list(model.classes_)
    if 1 in classes:
        pos_idx = classes.index(1)
    else:
        # Fallback: assume the "positive" class is the last one
        pos_idx = -1
    return proba[:, pos_idx]

# Align feature columns to what the model was trained on (handles missing/extra columns)
if hasattr(rf_scores, 'feature_names_in_'):
    train_cols = list(rf_scores.feature_names_in_)
    X_test_aligned = X.reindex(columns=train_cols, fill_value=0)
else:
    # Older sklearn: assume columns already match
    X_test_aligned = X.copy()

# Predict probabilities
p_scores = _pos_class_proba(rf_scores, X_test_aligned)
p_concedes = _pos_class_proba(rf_concedes, X_test_aligned)

print('Predicted probas shapes:', p_scores.shape, p_concedes.shape)

# Basic evaluation (only if both classes present)
def _eval(name, y_true, p):
    y_unique = set(pd.Series(y_true).unique().tolist())
    print(f"\n[{name}]")
    if len(y_unique) < 2:
        print('Only one class present in y; skipping AUC/logloss metrics. Classes:', y_unique)
        return
    print('ROC AUC:  ', roc_auc_score(y_true, p))
    print('PR AUC:   ', average_precision_score(y_true, p))
    print('Log loss: ', log_loss(y_true, p, labels=[0, 1]))
    print('Brier:    ', brier_score_loss(y_true, p))

_eval('scores', y_scores, p_scores)
_eval('concedes', y_concedes, p_concedes)

# Threshold-based metrics
threshold = 0.5
yhat_scores = (p_scores >= threshold).astype(int)
yhat_concedes = (p_concedes >= threshold).astype(int)

def _eval_threshold(name, y_true, y_pred):
    print(f"\n[{name}] (threshold={threshold})")
    print('Accuracy: ', accuracy_score(y_true, y_pred))
    print('Precision:', precision_score(y_true, y_pred, zero_division=0))
    print('Recall:   ', recall_score(y_true, y_pred, zero_division=0))
    print('F1:       ', f1_score(y_true, y_pred, zero_division=0))

_eval_threshold('scores:', y_scores, yhat_scores)
_eval_threshold('concedes:', y_concedes, yhat_concedes)

# Preview a few predictions alongside labels
df_pred = df_test[['game_id', 'action_id', 'scores', 'concedes']].copy()
df_pred['p_scores'] = p_scores
df_pred['p_concedes'] = p_concedes
df_pred['yhat_scores'] = yhat_scores
df_pred['yhat_concedes'] = yhat_concedes
display(df_pred.head(10))

Predicted probas shapes: (1711,) (1711,)

[scores]
ROC AUC:   1.0
PR AUC:    1.0
Log loss:  0.011800634987674
Brier:     0.0023819988310929277

[concedes]
ROC AUC:   1.0
PR AUC:    1.0
Log loss:  0.003146934842677171
Brier:     0.0006162478082992403

[scores:] (threshold=0.5)
Accuracy:  1.0
Precision: 1.0
Recall:    1.0
F1:        1.0

[concedes:] (threshold=0.5)
Accuracy:  1.0
Precision: 1.0
Recall:    1.0
F1:        1.0


,game_id,action_id,scores,concedes,p_scores,p_concedes,yhat_scores,yhat_concedes
0,3889182,0,False,False,0.005,0.0,0,0
1,3889182,1,False,False,0.000,0.0,0,0
2,3889182,2,False,False,0.000,0.0,0,0
3,3889182,3,False,False,0.005,0.0,0,0
4,3889182,4,False,False,0.000,0.0,0,0
5,3889182,5,False,False,0.005,0.0,0,0
6,3889182,6,False,False,0.020,0.0,0,0
7,3889182,7,False,False,0.000,0.0,0,0
8,3889182,8,False,False,0.000,0.0,0,0
9,3889182,9,False,False,0.000,0.0,0,0
